<a href="https://colab.research.google.com/github/khanhnk2010/ML2-Product-Recommendation-/blob/main/Recommender_system_FE_PCA%26logistics_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gc #để xoá object khỏi ram khi không cần nữa

from google.colab import drive
#drive ngáo lắm phải forece_remount
drive.mount("/content/drive", force_remount=True)
import shutil
import os
import time
import pyarrow.parquet as pq
import pyarrow as pa

pd.set_option('display.max_columns', None)
if hasattr(pd, 'options'):
    pd.options.mode.copy_on_write = True  #chỉ tạo copy của dataframe khi có hành động write->kiểm soát số lượng copy

Mounted at /content/drive


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, label_ranking_average_precision_score, classification_report, matthews_corrcoef
import joblib



In [ ]:
month_int = list(range(1,17))

In [ ]:
target = ["product_1","product_2",
              "product_3","product_4","product_5",
               "product_6","product_7","product_8","product_9","product_10","product_11",
               "product_12","product_13","product_14","product_15","product_16",
               "product_17","product_18","product_19","product_20","product_21",
               "product_22","product_23","product_24"]
target_cols = [f'{product}_last_0' for product in target[2:]]
removed_labels = [f'{product}_last_0' for product in target[:2]]


In [ ]:
customer_info_cols = ['customer_code', 'employee_index', 'customer_country', 'sex', 'age',
       'new_index', 'seniority_months', 'primary_customer', 'month_start_type',
       'relation_type', 'residence_index', 'foreigner_index', 'join_channel',
       'province_code', 'activity_index', 'gross_househole_income', 'segment']

In [ ]:
merged_train_dataset = '/content/drive/My Drive/bank_data/merged_train_dataset/'
trend_pca_parquet = '/content/drive/My Drive/bank_data/trend_pca.parquet'
new_buy_pca_parquet = '/content/drive/My Drive/bank_data/new_buy_pca.parquet'
pca_multilabel_parquet = '/content/drive/My Drive/bank_data/pca_multilabel.parquet'
pca_multilabel_val_parquet = '/content/drive/My Drive/bank_data/pca_multilabel_val.parquet'

In [ ]:
merged_train_dataset = '/content/drive/My Drive/bank_data/merged_train_dataset/'
df_list = []
trend_cols = [f'product_{label+1}'+f'_diff_last_{month}' for label in list(range(0,24)) for month in list(range(5,0,-1))]
for i in range(1, 1+9):
  filepath = f"chunk_{i}.parquet"
  df = pd.read_parquet(merged_train_dataset+filepath)
  df = df[trend_cols]
  df_list.append(df)
df_trend_pca = pd.concat(df_list, ignore_index=True)
for df in df_list:
  del df
  gc.collect()
display(df_trend_pca.head())


,product_1_diff_last_5,product_1_diff_last_4,product_1_diff_last_3,product_1_diff_last_2,product_1_diff_last_1,product_2_diff_last_5,product_2_diff_last_4,product_2_diff_last_3,product_2_diff_last_2,product_2_diff_last_1,product_3_diff_last_5,product_3_diff_last_4,product_3_diff_last_3,product_3_diff_last_2,product_3_diff_last_1,product_4_diff_last_5,product_4_diff_last_4,product_4_diff_last_3,product_4_diff_last_2,product_4_diff_last_1,product_5_diff_last_5,product_5_diff_last_4,product_5_diff_last_3,product_5_diff_last_2,product_5_diff_last_1,product_6_diff_last_5,product_6_diff_last_4,product_6_diff_last_3,product_6_diff_last_2,product_6_diff_last_1,product_7_diff_last_5,product_7_diff_last_4,product_7_diff_last_3,product_7_diff_last_2,product_7_diff_last_1,product_8_diff_last_5,product_8_diff_last_4,product_8_diff_last_3,product_8_diff_last_2,product_8_diff_last_1,product_9_diff_last_5,product_9_diff_last_4,product_9_diff_last_3,product_9_diff_last_2,product_9_diff_last_1,product_10_diff_last_5,product_10_diff_last_4,product_10_diff_last_3,product_10_diff_last_2,product_10_diff_last_1,product_11_diff_last_5,product_11_diff_last_4,product_11_diff_last_3,product_11_diff_last_2,product_11_diff_last_1,product_12_diff_last_5,product_12_diff_last_4,product_12_diff_last_3,product_12_diff_last_2,product_12_diff_last_1,product_13_diff_last_5,product_13_diff_last_4,product_13_diff_last_3,product_13_diff_last_2,product_13_diff_last_1,product_14_diff_last_5,product_14_diff_last_4,product_14_diff_last_3,product_14_diff_last_2,product_14_diff_last_1,product_15_diff_last_5,product_15_diff_last_4,product_15_diff_last_3,product_15_diff_last_2,product_15_diff_last_1,product_16_diff_last_5,product_16_diff_last_4,product_16_diff_last_3,product_16_diff_last_2,product_16_diff_last_1,product_17_diff_last_5,product_17_diff_last_4,product_17_diff_last_3,product_17_diff_last_2,product_17_diff_last_1,product_18_diff_last_5,product_18_diff_last_4,product_18_diff_last_3,product_18_diff_last_2,product_18_diff_last_1,product_19_diff_last_5,product_19_diff_last_4,product_19_diff_last_3,product_19_diff_last_2,product_19_diff_last_1,product_20_diff_last_5,product_20_diff_last_4,product_20_diff_last_3,product_20_diff_last_2,product_20_diff_last_1,product_21_diff_last_5,product_21_diff_last_4,product_21_diff_last_3,product_21_diff_last_2,product_21_diff_last_1,product_22_diff_last_5,product_22_diff_last_4,product_22_diff_last_3,product_22_diff_last_2,product_22_diff_last_1,product_23_diff_last_5,product_23_diff_last_4,product_23_diff_last_3,product_23_diff_last_2,product_23_diff_last_1,product_24_diff_last_5,product_24_diff_last_4,product_24_diff_last_3,product_24_diff_last_2,product_24_diff_last_1
0,-0.000141,-0.000002,0.0,0.0,0.0,-0.000029,-6.896347e-09,-5.761939e-09,-0.000002,0.0,-0.775652,-0.000172,-0.000582,0.000012,0.00655,-0.000466,0.000008,-0.000008,-0.000008,0.000003,-0.091576,-0.000193,0.000338,-0.00014,-0.000089,-0.012296,0.000036,-0.00001,0.000023,-0.000046,-0.012685,0.000007,-0.000123,-0.000027,0.000063,-0.171001,0.000045,-0.000016,0.000027,-0.000122,-0.058043,-0.000011,0.000009,6.679350e-07,-0.000033,-0.001055,-0.000029,-0.000415,-0.000082,0.000105,-0.002563,-0.000016,0.000005,-0.000002,0.000013,-0.056855,0.000186,-0.000048,0.000195,0.000121,-0.09459,0.000193,-0.000197,-0.000011,-0.000389,-0.021645,0.000192,-0.000126,-0.000134,-0.000274,-0.008002,0.000003,-0.000005,0.000021,-0.000021,-0.01194,-0.000008,-0.000003,-1.276146e-07,-0.000005,-0.003349,0.000006,0.000008,-0.000011,-0.000005,-0.058815,0.000007,0.001045,-0.000851,0.004112,-0.051771,0.002582,-0.000745,-0.000868,0.000871,-0.032244,-0.000096,-0.000022,0.000023,-0.000087,-0.005199,-0.000006,0.000002,-0.000002,-0.000002,-0.053751,-0.001878,-0.002931,0.001078,0.002725,-0.060742,-0.001893,0.001803,-0.003682,0.007614,-0.140615,0.001375,-0.001315,-0.002081,0.00307
1,-0.000141,-0.000002,0.0,0.0,0.0,-0.000029,-6.896347e-09,-5.761939e-09,-0.000002,0.0,-0.775652,-0.000172,-0.000582,0.000012,0.00655,-

In [ ]:
display(df_trend_pca.info())
display(df_trend_pca.describe().style.background_gradient())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7637019 entries, 0 to 7637018
Columns: 120 entries, product_1_diff_last_5 to product_24_diff_last_1
dtypes: float32(120)
memory usage: 3.4 GB


None

,product_1_diff_last_5,product_1_diff_last_4,product_1_diff_last_3,product_1_diff_last_2,product_1_diff_last_1,product_2_diff_last_5,product_2_diff_last_4,product_2_diff_last_3,product_2_diff_last_2,product_2_diff_last_1,product_3_diff_last_5,product_3_diff_last_4,product_3_diff_last_3,product_3_diff_last_2,product_3_diff_last_1,product_4_diff_last_5,product_4_diff_last_4,product_4_diff_last_3,product_4_diff_last_2,product_4_diff_last_1,product_5_diff_last_5,product_5_diff_last_4,product_5_diff_last_3,product_5_diff_last_2,product_5_diff_last_1,product_6_diff_last_5,product_6_diff_last_4,product_6_diff_last_3,product_6_diff_last_2,product_6_diff_last_1,product_7_diff_last_5,product_7_diff_last_4,product_7_diff_last_3,product_7_diff_last_2,product_7_diff_last_1,product_8_diff_last_5,product_8_diff_last_4,product_8_diff_last_3,product_8_diff_last_2,product_8_diff_last_1,product_9_diff_last_5,product_9_diff_last_4,product_9_diff_last_3,product_9_diff_last_2,product_9_diff_last_1,product_10_diff_last_5,product_10_diff_last_4,product_10_diff_last_3,product_10_diff_last_2,product_10_diff_last_1,product_11_diff_last_5,product_11_diff_last_4,product_11_diff_last_3,product_11_diff_last_2,product_11_diff_last_1,product_12_diff_last_5,product_12_diff_last_4,product_12_diff_last_3,product_12_diff_last_2,product_12_diff_last_1,product_13_diff_last_5,product_13_diff_last_4,product_13_diff_last_3,product_13_diff_last_2,product_13_diff_last_1,product_14_diff_last_5,product_14_diff_last_4,product_14_diff_last_3,product_14_diff_last_2,product_14_diff_last_1,product_15_diff_last_5,product_15_diff_last_4,product_15_diff_last_3,product_15_diff_last_2,product_15_diff_last_1,product_16_diff_last_5,product_16_diff_last_4,product_16_diff_last_3,product_16_diff_last_2,product_16_diff_last_1,product_17_diff_last_5,product_17_diff_last_4,product_17_diff_last_3,product_17_diff_last_2,product_17_diff_last_1,product_18_diff_last_5,product_18_diff_last_4,product_18_diff_last_3,product_18_diff_last_2,product_18_diff_last_1,product_19_diff_last_5,product_19_diff_last_4,product_19_diff_last_3,product_19_diff_last_2,product_19_diff_last_1,product_20_diff_last_5,product_20_diff_last_4,product_20_diff_last_3,product_20_diff_last_2,product_20_diff_last_1,product_21_diff_last_5,product_21_diff_last_4,product_21_diff_last_3,product_21_diff_last_2,product_21_diff_last_1,product_22_diff_last_5,product_22_diff_last_4,product_22_diff_last_3,product_22_diff_last_2,product_22_diff_last_1,product_23_diff_last_5,product_23_diff_last_4,product_23_diff_last_3,product_23_diff_last_2,product_23_diff_last_1,product_24_diff_last_5,product_24_diff_last_4,product_24_diff_last_3,product_24_diff_last_2,product_24_diff_last_1
count,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,763

In [ ]:
trend_pca = PCA(n_components=10)
trend_pca.fit(df_trend_pca)
cum_vars = np.cumsum(trend_pca.explained_variance_ratio_)
for i, var in enumerate(cum_vars):
  print(f"Component: {i+1}: Cumulative Variance: {var:.3f}")

Component: 1: Cumulative Variance: 0.992
Component: 2: Cumulative Variance: 0.996
Component: 3: Cumulative Variance: 0.997
Component: 4: Cumulative Variance: 0.998
Component: 5: Cumulative Variance: 0.999
Component: 6: Cumulative Variance: 0.999
Component: 7: Cumulative Variance: 1.000
Component: 8: Cumulative Variance: 1.000
Component: 9: Cumulative Variance: 1.000
Component: 10: Cumulative Variance: 1.000


In [ ]:
pca_features = [f'trend_pca_{i}' for i in range(1, 11)]
print(pca_features)
df_trend_pca = pd.DataFrame(trend_pca.transform(df_trend_pca), columns=pca_features)

['trend_pca_1', 'trend_pca_2', 'trend_pca_3', 'trend_pca_4', 'trend_pca_5', 'trend_pca_6', 'trend_pca_7', 'trend_pca_8', 'trend_pca_9', 'trend_pca_10']


In [ ]:
display(df_trend_pca.head())

,trend_pca_1,trend_pca_2,trend_pca_3,trend_pca_4,trend_pca_5,trend_pca_6,trend_pca_7,trend_pca_8,trend_pca_9,trend_pca_10
0,-0.762762,-0.000788,-0.000419,0.00049,0.00011,-0.00011,0.000054,0.000018,-0.000002,-0.000005
1,-0.762762,-0.000788,-0.000419,0.00049,0.00011,-0.00011,0.000054,0.000018,-0.000002,-0.000005
2,-0.762762,-0.000788,-0.000419,0.00049,0.00011,-0.00011,0.000054,0.000018,-0.000002,-0.000005
3,-0.762762,-0.000788,-0.000419,0.00049,0.00011,-0.00011,0.000054,0.000018,-0.000002,-0.000005
4,-0.762762,-0.000788,-0.000419,0.00049,0.00011,-0.00011,0.000054,0.000018,-0.000002,-0.000005


In [ ]:
trend_pca_parquet = '/content/drive/My Drive/bank_data/trend_pca.parquet'
df_trend_pca.to_parquet(trend_pca_parquet, index=False)
del df_trend_pca
gc.collect()

1818

In [ ]:
df_trend_pca = pd.read_parquet(trend_pca_parquet)
for col in df_trend_pca.columns:
  print(df_trend_pca[col].min())
  print(df_trend_pca[col].max())

-0.7627615332603455
0.08237127214670181
-0.02263413555920124
0.022832317277789116
-0.0122755765914917
0.013954190537333488
-0.008563435636460781
0.012826679274439812
-0.011106714606285095
0.00713595375418663
-0.00897944439202547
0.007927043363451958
-0.007475723512470722
0.0053549278527498245
-0.005401367321610451
0.0036547549534589052
-7.906035898486152e-05
0.00012871046783402562
-8.459472155664116e-06
9.867508197203279e-06


In [ ]:
merged_train_dataset = '/content/drive/My Drive/bank_data/merged_train_dataset/'
df_list = []
new_buy_cols = [f'product_{label+1}'+f'_last_{month}' for label in list(range(0,24)) for month in list(range(6,0,-1))]
for i in range(1, 1+9):
  filepath = f"chunk_{i}.parquet"
  df = pd.read_parquet(merged_train_dataset+filepath)
  for col in new_buy_cols:
    df[col] = df[col].astype(np.int8)
  df.drop(columns=[col for col in df.columns if col not in new_buy_cols], inplace=True)
  new_buy_pca = PCA(n_components=50)
  new_buy_pca.fit(df)
  pca_cols = [f'new_buy_pca_{i+1}' for i in range(0,50)]
  df = pd.DataFrame(new_buy_pca.transform(df), columns=pca_cols)
  df_list.append(df)
df_new_buy_pca = pd.concat(df_list, ignore_index=True)
for df in df_list:
  del df
  gc.collect()
del df_list
gc.collect()
display(df_new_buy_pca.head())


,new_buy_pca_1,new_buy_pca_2,new_buy_pca_3,new_buy_pca_4,new_buy_pca_5,new_buy_pca_6,new_buy_pca_7,new_buy_pca_8,new_buy_pca_9,new_buy_pca_10,new_buy_pca_11,new_buy_pca_12,new_buy_pca_13,new_buy_pca_14,new_buy_pca_15,new_buy_pca_16,new_buy_pca_17,new_buy_pca_18,new_buy_pca_19,new_buy_pca_20,new_buy_pca_21,new_buy_pca_22,new_buy_pca_23,new_buy_pca_24,new_buy_pca_25,new_buy_pca_26,new_buy_pca_27,new_buy_pca_28,new_buy_pca_29,new_buy_pca_30,new_buy_pca_31,new_buy_pca_32,new_buy_pca_33,new_buy_pca_34,new_buy_pca_35,new_buy_pca_36,new_buy_pca_37,new_buy_pca_38,new_buy_pca_39,new_buy_pca_40,new_buy_pca_41,new_buy_pca_42,new_buy_pca_43,new_buy_pca_44,new_buy_pca_45,new_buy_pca_46,new_buy_pca_47,new_buy_pca_48,new_buy_pca_49,new_buy_pca_50
0,-0.037015,-0.296484,0.215059,0.002599,0.132069,1.023087,-0.396872,-0.345838,0.905180,0.691392,-0.091519,-0.005562,-0.354078,-0.069495,0.035557,0.021250,0.018959,0.005323,-0.011319,-0.006932,0.039301,-0.086261,-0.009202,0.013343,-0.001564,-0.003271,0.007245,0.045261,0.021684,-0.016697,-0.248414,0.007246,0.084145,0.817730,0.253536,-0.050754,-0.339999,-0.013564,0.006984,-0.003281,0.002220,0.004463,-0.008334,-0.008228,0.003840,-0.009866,0.009343,0.002451,-0.002263,0.001817
1,-2.253485,-0.528356,0.063730,0.148208,-0.241081,0.428077,-0.232884,-0.753045,0.487606,-0.224950,-0.238442,-0.076874,0.080875,-0.053356,-0.031212,-0.020820,-0.046400,-0.030254,-0.003008,-0.081435,-0.314938,0.808490,0.285544,0.082283,-0.010233,0.007458,-0.035577,-0.108030,0.009873,-0.017466,-0.043801,-0.117338,0.031079,-0.052212,0.010548,-0.023725,0.021815,0.013767,-0.051291,0.000988,0.002270,0.009671,-0.010306,-0.003220,-0.003032,0.000629,0.001704,-0.004007,0.003311,-0.003319
2,-1.749251,-0.353408,0.172518,1.091526,0.558655,0.315777,-0.032972,0.894516,0.387281,0.646528,0.300936,-0.571343,-0.379017,-0.012301,0.833287,0.078959,0.167015,-0.032434,-0.022573,0.087251,-0.058885,-0.148865,0.053572,0.132811,-0.049903,0.018400,-0.015966,-0.132456,0.031494,0.010054,-0.053519,-0.035619,0.014867,-0.068469,0.010792,-0.014629,0.032285,0.047582,-0.008610,-0.006974,0.004074,-0.008390,0.022168,-0.031048,-0.001748,-0.007407,-0.005412,-0.001444,0.001423,0.029624
3,-0.288182,0.368278,-0.462702,0.498911,0.078220,0.453328,0.673477,0.409683,0.052331,0.859238,-0.161005,-0.022020,-0.417411,-0.074414,-0.047169,-0.011610,0.009817,-0.031913,-0.001343,-0.037272,-0.016281,-0.059065,-0.050290,-0.038911,-0.001571,-0.008760,0.021045,-0.031284,-0.026929,0.020122,-0.018863,-0.007163,-0.136228,-0.000157,0.000052,-0.009559,-0.002259,0.018301,-0.003806,-0.020630,-0.003097,0.004704,-0.011940,-0.013913,0.003369,-0.002988,0.003910,0.001561,-0.005973,-0.001288
4,-1.901203,-1.124044,0.878941,0.594674,-0.602289,0.361316,-0.050669,0.792850,0.186778,0.571487,-0.189984,-0.086249,-0.447749,-0.086285,0.016051,0.005777,-0.009889,-0.030850,-0.005050,-0.007490,0.098414,-0.095059,-0.031416,0.041376,-0.118455,0.116968,-0.080327,-0.129437,0.048354,-0.014752,-0.049908,-0.056360,0.074902,-0.078969,0.009885,-0.003042,0.008954,0.273551,-0.035031,0.039666,0.624873,-0.132103,-0.256880,0.107400,0.062105,0.297470,-0.374755,-0.410910,0.034174,-0.022147


In [ ]:
display(df_new_buy_pca.info())
display(df_new_buy_pca.describe().style.background_gradient())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7637019 entries, 0 to 7637018
Columns: 144 entries, product_1_last_6 to product_24_last_1
dtypes: int8(144)
memory usage: 1.0 GB


None

,product_1_last_6,product_2_last_6,product_3_last_6,product_4_last_6,product_5_last_6,product_6_last_6,product_7_last_6,product_8_last_6,product_9_last_6,product_10_last_6,product_11_last_6,product_12_last_6,product_13_last_6,product_14_last_6,product_15_last_6,product_16_last_6,product_17_last_6,product_18_last_6,product_19_last_6,product_20_last_6,product_21_last_6,product_22_last_6,product_23_last_6,product_24_last_6,product_1_last_5,product_2_last_5,product_3_last_5,product_4_last_5,product_5_last_5,product_6_last_5,product_7_last_5,product_8_last_5,product_9_last_5,product_10_last_5,product_11_last_5,product_12_last_5,product_13_last_5,product_14_last_5,product_15_last_5,product_16_last_5,product_17_last_5,product_18_last_5,product_19_last_5,product_20_last_5,product_21_last_5,product_22_last_5,product_23_last_5,product_24_last_5,product_1_last_4,product_2_last_4,product_3_last_4,product_4_last_4,product_5_last_4,product_6_last_4,product_7_last_4,product_8_last_4,product_9_last_4,product_10_last_4,product_11_last_4,product_12_last_4,product_13_last_4,product_14_last_4,product_15_last_4,product_16_last_4,product_17_last_4,product_18_last_4,product_19_last_4,product_20_last_4,product_21_last_4,product_22_last_4,product_23_last_4,product_24_last_4,product_1_last_3,product_2_last_3,product_3_last_3,product_4_last_3,product_5_last_3,product_6_last_3,product_7_last_3,product_8_last_3,product_9_last_3,product_10_last_3,product_11_last_3,product_12_last_3,product_13_last_3,product_14_last_3,product_15_last_3,product_16_last_3,product_17_last_3,product_18_last_3,product_19_last_3,product_20_last_3,product_21_last_3,product_22_last_3,product_23_last_3,product_24_last_3,product_1_last_2,product_2_last_2,product_3_last_2,product_4_last_2,product_5_last_2,product_6_last_2,product_7_last_2,product_8_last_2,product_9_last_2,product_10_last_2,product_11_last_2,product_12_last_2,product_13_last_2,product_14_last_2,product_15_last_2,product_16_last_2,product_17_last_2,product_18_last_2,product_19_last_2,product_20_last_2,product_21_last_2,product_22_last_2,product_23_last_2,product_24_last_2,product_1_last_1,product_2_last_1,product_3_last_1,product_4_last_1,product_5_last_1,product_6_last_1,product_7_last_1,product_8_last_1,product_9_last_1,product_10_last_1,product_11_last_1,product_12_last_1,product_13_last_1,product_14_last_1,product_15_last_1,product_16_last_1,product_17_last_1,product_18_last_1,product_19_last_1,product_20_last_1,product_21_last_1,product_22_last_1,product_23_last_1,product_24_last_1
count,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,7637019.000000,

In [ ]:
new_buy_pca = PCA(n_components=10)
new_buy_pca.fit(df_new_buy_pca)
cum_vars = np.cumsum(new_buy_pca.explained_variance_ratio_)
for i, var in enumerate(cum_vars):
  print(f"Component: {i+1}: Cumulative Variance: {var:.3f}")

In [ ]:
new_buy_pca_parquet = '/content/drive/My Drive/bank_data/new_buy_pca.parquet'
df_new_buy_pca.to_parquet(new_buy_pca_parquet, index=False)
del df_new_buy_pca
gc.collect()

0

In [ ]:
df_new_buy_pca = pd.read_parquet(new_buy_pca_parquet)
for col in df_new_buy_pca.columns:
  print(df_new_buy_pca[col].min())
  print(df_new_buy_pca[col].max())

-2.648378756935906
2.2944658702240064
-1.1935011883942384
2.159778725029182
-1.0728773526449489
1.941872942470442
-1.75976150852106
1.8419253423767148
-1.4765994086820657
2.1381930516134835
-1.5634915234390732
1.8357326902943374
-1.5325004875051627
1.9152122609094588
-1.8447610386839577
1.9121317856734532
-1.4789066969623499
1.865409234836055
-1.6366582639920468
1.6125285380252212
-1.6780290208230253
2.157317711453914
-1.353459313769055
1.838877769253882
-1.6252111319244973
1.6920522932223856
-1.368021355168389
1.8110888020783262
-1.3558380413232831
1.5972706143408193
-0.9991501793905295
1.830013684081221
-1.450356938415734
1.6661283351683431
-1.3238921877131493
2.0503768496240538
-1.0835286142745384
1.6523124032251497
-1.0327061006398177
1.1874212811085743
-1.2068088667724768
1.285306736248848
-0.973640100429754
1.2594539790742525
-1.0408705035799246
1.1412682066226993
-0.8192996832564093
1.6025782887626236
-1.4069013615119095
1.0164075604414455
-0.927804711172965
1.534782006531196
-1

In [ ]:
df_list = []

for i in range(1, 1+9):
  filepath = merged_train_dataset + f"chunk_{i}.parquet"
  df = pd.read_parquet(filepath)
  df.drop(columns=[col for col in df.columns if col not in customer_info_cols+['month_sin', 'month_cos']+target_cols], inplace=True)
  for col in target_cols:
    df[col] = df[col].astype(np.int8)
  df_list.append(df)
df_customer_info_target_cols = pd.concat(df_list, ignore_index=True)
print("yes?")
for df in df_list:
  del df
  gc.collect()
del df_list
gc.collect()

yes?


0

In [ ]:
df_trend_pca = pd.read_parquet(trend_pca_parquet)
for col in df_trend_pca.columns:
  df_trend_pca[col] = df_trend_pca[col].astype(np.float32)

In [ ]:
df_new_buy_pca = pd.read_parquet(new_buy_pca_parquet)
for col in df_new_buy_pca.columns:
  df_new_buy_pca[col] = df_new_buy_pca[col].astype(np.float32)

In [ ]:
df_customer_info_target_cols = pd.concat([df_customer_info_target_cols, df_trend_pca], axis=1, ignore_index=False)
del df_trend_pca
gc.collect()

0

In [ ]:
df_customer_info_target_cols.to_parquet(pca_multilabel_parquet, index=False)
del df_customer_info_target_cols
gc.collect()

0

In [ ]:
df = pd.read_parquet(pca_multilabel_parquet)
df = pd.concat([df, df_new_buy_pca], axis=1, ignore_index=False)
del df_new_buy_pca
gc.collect()

0

In [ ]:
display(df.head())

,customer_code,employee_index,customer_country,sex,age,new_index,seniority_months,primary_customer,month_start_type,relation_type,residence_index,foreigner_index,join_channel,province_code,activity_index,gross_househole_income,segment,month_sin,month_cos,product_1_last_0,product_2_last_0,product_3_last_0,product_4_last_0,product_5_last_0,product_6_last_0,product_7_last_0,product_8_last_0,product_9_last_0,product_10_last_0,product_11_last_0,product_12_last_0,product_13_last_0,product_14_last_0,product_15_last_0,product_16_last_0,product_17_last_0,product_18_last_0,product_19_last_0,product_20_last_0,product_21_last_0,product_22_last_0,product_23_last_0,product_24_last_0,trend_pca_1,trend_pca_2,trend_pca_3,trend_pca_4,trend_pca_5,trend_pca_6,trend_pca_7,trend_pca_8,trend_pca_9,trend_pca_10,new_buy_pca_1,new_buy_pca_2,new_buy_pca_3,new_buy_pca_4,new_buy_pca_5,new_buy_pca_6,new_buy_pca_7,new_buy_pca_8,new_buy_pca_9,new_buy_pca_10,new_buy_pca_11,new_buy_pca_12,new_buy_pca_13,new_buy_pca_14,new_buy_pca_15,new_buy_pca_16,new_buy_pca_17,new_buy_pca_18,new_buy_pca_19,new_buy_pca_20,new_buy_pca_21,new_buy_pca_22,new_buy_pca_23,new_buy_pca_24,new_buy_pca_25,new_buy_pca_26,new_buy_pca_27,new_buy_pca_28,new_buy_pca_29,new_buy_pca_30,new_buy_pca_31,new_buy_pca_32,new_buy_pca_33,new_buy_pca_34,new_buy_pca_35,new_buy_pca_36,new_buy_pca_37,new_buy_pca_38,new_buy_pca_39,new_buy_pca_40,new_buy_pca_41,new_buy_pca_42,new_buy_pca_43,new_buy_pca_44,new_buy_pca_45,new_buy_pca_46,new_buy_pca_47,new_buy_pca_48,new_buy_pca_49,new_buy_pca_50
0,15889,F,0.00169,1,56,0,245,1,1.0,A,1,0,0.002324,0.002235,1,326124.90,01 - TOP,1.224647e-16,-1.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-0.762762,-0.000788,-0.000419,0.00049,0.00011,-0.00011,0.000054,0.000018,-0.000002,-0.000005,-0.037015,-0.296484,0.215059,0.002599,0.132069,1.023087,-0.396872,-0.345838,0.905180,0.691392,-0.091519,-0.005562,-0.354078,-0.069495,0.035557,0.021250,0.018959,0.005323,-0.011319,-0.006932,0.039301,-0.086261,-0.009202,0.013343,-0.001564,-0.003271,0.007245,0.045261,0.021684,-0.016697,-0.248414,0.007246,0.084145,0.817730,0.253536,-0.050754,-0.339999,-0.013564,0.006984,-0.003281,0.002220,0.004463,-0.008334,-0.008228,0.003840,-0.009866,0.009343,0.002451,-0.002263,0.001817
1,15890,A,0.00169,1,62,0,246,1,1.0,A,1,0,0.002324,0.002235,1,71461.20,02 - PARTICULARES,1.224647e-16,-1.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-0.762762,-0.000788,-0.000419,0.00049,0.00011,-0.00011,0.000054,0.000018,-0.000002,-0.000005,-2.253484,-0.528356,0.063730,0.148208,-0.241081,0.428077,-0.232884,-0.753045,0.487606,-0.224950,-0.238442,-0.076874,0.080875,-0.053356,-0.031212,-0.020820,-0.046400,-0.030254,-0.003008,-0.081435,-0.314938,0.808490,0.285544,0.082283,-0.010233,0.007458,-0.035577,-0.108030,0.009873,-0.017466,-0.043801,-0.117338,0.031079,-0.052212,0.010548,-0.023725,0.021815,0.013767,-0.051291,0.000988,0.002270,0.009671,-0.010306,-0.003220,-0.003032,0.000629,0.001704,-0.004007,0.003311,-0.003319
2,15892,F,0.00169,0,61,0,246,1,1.0,A,1,0,0.002324,0.002235,1,430477.41,01 - TOP,1.224647e-16,-1.0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,-0.762762,-0.000788,-0.000419,0.00049,0.00011,-0.00011,0.000054,0.000018,-0.000002,-0.000005,-1.749251,-0.353408,0.172518,1.091526,0.558655,0.315777,-0.032972,0.894516,0.387281,0.646528,0.300936,-0.571343,-0.379017,-0.012301,0.833287,0.078959,0.167015,-0.032434,-0.022573,0.087251,-0.058885,-0.148865,0.053572,0.132811,-0.049903,0.018400,-0.015966,-0.132456,0.031494,0.010054,-0.053519,-0.035619,0.014867,-0.068469,0.010792,-0.014629,0.032285,0.047582,-0.008610,-0.006974,0.004074,-0.008390,0.022168,-0.031048,-0.001748,-0.007407,-0.005412,-0.001444,0.001423,0.029624
3,15893,N,0.00169,1,62,0,246,1,1.0,A,1,0,0.002324,0.002235,1,430477.41,02 - PARTICULARES,1.224647e-16,-1.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-0.762762,-0.000788,-0.000419,0.00049,0.00011,-0.00011,0.000054,0.000018,-0.000002,-0.000005,-0.288182,0.368278,-0.462702,0.498911,0.078220,0.453328,0.

In [ ]:
df.to_parquet(pca_multilabel_parquet, index=False)
del df
gc.collect()

0

In [ ]:
del df_trend_pca
gc.collect()

0

In [ ]:
merged_valid_dataset = '/content/drive/My Drive/bank_data/merged_valid_dataset/'
for filepath in os.listdir(merged_valid_dataset):
  df = pd.read_parquet(merged_valid_dataset+filepath)

In [ ]:
df_trend_pca = df[trend_cols].copy()
df.drop(columns=trend_cols, inplace=True)
pca_features = [f'trend_pca_{i}' for i in range(1, 11)]
df_trend_pca = pd.DataFrame(trend_pca.transform(df_trend_pca), columns=pca_features)
df = pd.concat([df, df_trend_pca], axis=1, ignore_index=False)
del df_trend_pca
gc.collect()

0

In [ ]:
new_buy_cols = [f'product_{label+1}'+f'_last_{month}' for label in list(range(0,24)) for month in list(range(6,0,-1))]
df_new_buy_pca = df[new_buy_cols].copy()
df.drop(columns=new_buy_cols, inplace=True)
new_buy_pca = PCA(n_components=50)
pca_cols = [f'new_buy_pca_{i+1}' for i in range(0,50)]
df_new_buy_pca = pd.DataFrame(new_buy_pca.fit_transform(df_new_buy_pca), columns=pca_cols)
df = pd.concat([df, df_new_buy_pca], axis=1, ignore_index=False)
del df_new_buy_pca
gc.collect()

0

In [ ]:
df.to_parquet(pca_multilabel_val_parquet, index=False)
del df
gc.collect()

0

In [ ]:
df = pd.read_parquet(pca_multilabel_parquet)


In [ ]:
ordinal_encode_dict = {}
for col in df.columns:
  if df[col].dtype.name == 'category':
    print(col)
    print(df[col].unique())
    ordinal_encode_dict[col] = {value: i for i, value in enumerate(df[col].unique())}
print(ordinal_encode_dict)

employee_index
['F', 'A', 'N', 'B', 'S']
Categories (5, object): ['A', 'B', 'F', 'N', 'S']
month_start_type
['1.0', '3.0', '2.0', 'P', '4.0']
Categories (5, object): ['1.0', '2.0', '3.0', '4.0', 'P']
relation_type
['A', 'I', 'P', 'R']
Categories (4, object): ['A', 'I', 'P', 'R']
segment
['01 - TOP', '02 - PARTICULARES', '03 - UNIVERSITARIO']
Categories (3, object): ['01 - TOP', '02 - PARTICULARES', '03 - UNIVERSITARIO']
{'employee_index': {'F': 0, 'A': 1, 'N': 2, 'B': 3, 'S': 4}, 'month_start_type': {'1.0': 0, '3.0': 1, '2.0': 2, 'P': 3, '4.0': 4}, 'relation_type': {'A': 0, 'I': 1, 'P': 2, 'R': 3}, 'segment': {'01 - TOP': 0, '02 - PARTICULARES': 1, '03 - UNIVERSITARIO': 2}}


In [ ]:
for col in ordinal_encode_dict.keys():
  df[col] = df[col].map(ordinal_encode_dict[col])
  df[col] = df[col].astype(np.int8)

In [ ]:
display(df.columns)
display(df.shape)
display(df.info())

Index(['customer_code', 'employee_index', 'customer_country', 'sex', 'age',
       'new_index', 'seniority_months', 'primary_customer', 'month_start_type',
       'relation_type',
       ...
       'new_buy_pca_41', 'new_buy_pca_42', 'new_buy_pca_43', 'new_buy_pca_44',
       'new_buy_pca_45', 'new_buy_pca_46', 'new_buy_pca_47', 'new_buy_pca_48',
       'new_buy_pca_49', 'new_buy_pca_50'],
      dtype='object', length=103)

(7637019, 103)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7637019 entries, 0 to 7637018
Columns: 103 entries, customer_code to new_buy_pca_50
dtypes: float32(65), float64(1), int16(1), int32(1), int8(35)
memory usage: 2.2 GB


None

In [ ]:
class_weight_dict = {}
n_rows = len(df)
for col in target_cols:
  pos_weight = n_rows / len(df[df[col]==1]) / 2
  neg_weight = n_rows / len(df[df[col]==0]) / 2
  class_weight_dict[col] = {0: neg_weight, 1: pos_weight}
joblib.dump(class_weight_dict, '/content/drive/My Drive/bank_data/class_weight_dict.joblib')
print(class_weight_dict)

{'product_3_last_0': {0: 0.5033146717871904, 1: 75.92224873247838}, 'product_4_last_0': {0: 0.500004582982026, 1: 54550.135714285716}, 'product_5_last_0': {0: 0.5008714236019839, 1: 287.38688191465343}, 'product_6_last_0': {0: 0.5000495006523479, 1: 5050.938492063492}, 'product_7_last_0': {0: 0.5002974787480553, 1: 840.89616824488}, 'product_8_last_0': {0: 0.5003490051747371, 1: 716.8217570865403}, 'product_9_last_0': {0: 0.5001973406884931, 1: 1267.3446730833057}, 'product_10_last_0': {0: 0.5002409823222862, 1: 1037.9204946996467}, 'product_11_last_0': {0: 0.50003129689214, 1: 7988.51359832636}, 'product_12_last_0': {0: 0.5007088057393662, 1: 353.2059476459162}, 'product_13_last_0': {0: 0.5004154938787877, 1: 602.1935814540293}, 'product_14_last_0': {0: 0.5001018274724365, 1: 2455.633118971061}, 'product_15_last_0': {0: 0.5000127016135633, 1: 19683.038659793816}, 'product_16_last_0': {0: 0.5000309040198404, 1: 8090.0625}, 'product_17_last_0': {0: 0.5000081839554144, 1: 30548.076}, 'pr

In [ ]:
class_weight_dict = joblib.load('/content/drive/My Drive/bank_data/class_weight_dict.joblib')

In [ ]:
df['gross_househole_income'] = np.log1p(df['gross_househole_income'])
scale_cols = [col for col in df.columns if col not in target_cols+removed_labels+['customer_code']]
scale_dict = {}
for col in scale_cols:
  mean = df[col].mean()
  stddev = df[col].std()
  scale_dict[col] = [mean, stddev]

In [ ]:
no_buy = df[df[target_cols].sum(axis=1) == 0].index
print(len(no_buy))
yes_buy = df[df[target_cols].sum(axis=1) > 0].index
df_neg = df.iloc[no_buy].copy()
df_pos = df.iloc[yes_buy].copy()
del df
gc.collect()
neg_downsample_ratio = 0.2
df_neg = df_neg.sample(frac=neg_downsample_ratio, random_state=42)
df = pd.concat([df_pos, df_neg], ignore_index=True)
del df_pos, df_neg
gc.collect()
df = df.sample(frac=1.0, random_state=42)
print(df.shape)

7385005
(1729015, 103)


In [ ]:
y_train = df[target_cols].copy()
X_train = df.drop(columns=target_cols+removed_labels+['customer_code']).copy()
del df
gc.collect()
for col in scale_cols:
  mean, stddev = scale_dict[col]
  X_train[col] = (X_train[col] - scale_dict[col][0]) / scale_dict[col][1]

In [ ]:
df_val = pd.read_parquet(pca_multilabel_val_parquet)
display(df_val.shape)
for col in ordinal_encode_dict.keys():
  df_val[col] = df_val[col].map(ordinal_encode_dict[col])
  df_val[col] = df_val[col].astype(np.int8)
df_val['gross_househole_income'] = np.log1p(df_val['gross_househole_income'])
no_buy = df_val[df_val[target_cols].sum(axis=1) == 0].index
print(len(no_buy))
yes_buy = df_val[df_val[target_cols].sum(axis=1) > 0].index
df_neg = df_val.iloc[no_buy].copy()
df_pos = df_val.iloc[yes_buy].copy()
del df_val
gc.collect()
neg_downsample_ratio = 0.7
df_neg = df_neg.sample(frac=neg_downsample_ratio, random_state=42)
df_val = pd.concat([df_pos, df_neg], ignore_index=True)
del df_pos, df_neg
gc.collect()
df_val = df_val.sample(frac=1.0, random_state=42)
print(df_val.shape)

y_val = df_val[target_cols].copy()
X_val = df_val.drop(columns=target_cols+removed_labels+['customer_code']).copy()
del df_val
gc.collect()
X_val['gross_househole_income'] = np.log1p(X_val['gross_househole_income'])
for col in scale_cols:
  X_val[col] = (X_val[col] - scale_dict[col][0]) / scale_dict[col][1]
gc.collect()

(927541, 103)

901814
(656997, 103)


0

In [ ]:
print(X_train.shape)
print(X_val.shape)
print(y_train.shape)
print(y_val.shape)

(1729015, 78)
(656997, 78)
(1729015, 22)
(656997, 22)


In [ ]:
display(X_val.columns)
display(X_train.columns)
display(y_val.columns)
display(y_train.columns)

Index(['employee_index', 'customer_country', 'sex', 'age', 'new_index',
       'seniority_months', 'primary_customer', 'month_start_type',
       'relation_type', 'residence_index', 'foreigner_index', 'join_channel',
       'province_code', 'activity_index', 'gross_househole_income', 'segment',
       'month_sin', 'month_cos', 'trend_pca_1', 'trend_pca_2', 'trend_pca_3',
       'trend_pca_4', 'trend_pca_5', 'trend_pca_6', 'trend_pca_7',
       'trend_pca_8', 'trend_pca_9', 'trend_pca_10', 'new_buy_pca_1',
       'new_buy_pca_2', 'new_buy_pca_3', 'new_buy_pca_4', 'new_buy_pca_5',
       'new_buy_pca_6', 'new_buy_pca_7', 'new_buy_pca_8', 'new_buy_pca_9',
       'new_buy_pca_10', 'new_buy_pca_11', 'new_buy_pca_12', 'new_buy_pca_13',
       'new_buy_pca_14', 'new_buy_pca_15', 'new_buy_pca_16', 'new_buy_pca_17',
       'new_buy_pca_18', 'new_buy_pca_19', 'new_buy_pca_20', 'new_buy_pca_21',
       'new_buy_pca_22', 'new_buy_pca_23', 'new_buy_pca_24', 'new_buy_pca_25',
       'new_buy_pca_26'

Index(['employee_index', 'customer_country', 'sex', 'age', 'new_index',
       'seniority_months', 'primary_customer', 'month_start_type',
       'relation_type', 'residence_index', 'foreigner_index', 'join_channel',
       'province_code', 'activity_index', 'gross_househole_income', 'segment',
       'month_sin', 'month_cos', 'trend_pca_1', 'trend_pca_2', 'trend_pca_3',
       'trend_pca_4', 'trend_pca_5', 'trend_pca_6', 'trend_pca_7',
       'trend_pca_8', 'trend_pca_9', 'trend_pca_10', 'new_buy_pca_1',
       'new_buy_pca_2', 'new_buy_pca_3', 'new_buy_pca_4', 'new_buy_pca_5',
       'new_buy_pca_6', 'new_buy_pca_7', 'new_buy_pca_8', 'new_buy_pca_9',
       'new_buy_pca_10', 'new_buy_pca_11', 'new_buy_pca_12', 'new_buy_pca_13',
       'new_buy_pca_14', 'new_buy_pca_15', 'new_buy_pca_16', 'new_buy_pca_17',
       'new_buy_pca_18', 'new_buy_pca_19', 'new_buy_pca_20', 'new_buy_pca_21',
       'new_buy_pca_22', 'new_buy_pca_23', 'new_buy_pca_24', 'new_buy_pca_25',
       'new_buy_pca_26'

Index(['product_3_last_0', 'product_4_last_0', 'product_5_last_0',
       'product_6_last_0', 'product_7_last_0', 'product_8_last_0',
       'product_9_last_0', 'product_10_last_0', 'product_11_last_0',
       'product_12_last_0', 'product_13_last_0', 'product_14_last_0',
       'product_15_last_0', 'product_16_last_0', 'product_17_last_0',
       'product_18_last_0', 'product_19_last_0', 'product_20_last_0',
       'product_21_last_0', 'product_22_last_0', 'product_23_last_0',
       'product_24_last_0'],
      dtype='object')

Index(['product_3_last_0', 'product_4_last_0', 'product_5_last_0',
       'product_6_last_0', 'product_7_last_0', 'product_8_last_0',
       'product_9_last_0', 'product_10_last_0', 'product_11_last_0',
       'product_12_last_0', 'product_13_last_0', 'product_14_last_0',
       'product_15_last_0', 'product_16_last_0', 'product_17_last_0',
       'product_18_last_0', 'product_19_last_0', 'product_20_last_0',
       'product_21_last_0', 'product_22_last_0', 'product_23_last_0',
       'product_24_last_0'],
      dtype='object')

In [ ]:
n_features = 103
len_dtrain = 7637019
best_score = {}
for col in target_cols:
  best_score[col] = [[0.0, -1.0], None, None]


In [ ]:
best_score = joblib.load('/content/drive/My Drive/bank_data/best_score.joblib')

In [ ]:
import warnings

warnings.simplefilter("ignore", UserWarning)

In [ ]:
params = {
    'penalty': 'l2',
    'C': 1.0,
    'l1_ratio': 0.0,
    'tol': 1e-4,
    'class_weight': None,
    'random_state': 42,
    'solver': 'lbfgs',
    'max_iter': 100,
    'n_jobs': -1
}

param_grid = {
    'C': list(np.round(np.geomspace(1e-3,1, 20),5))+list(np.round(np.linspace(1e-3,1,20),5)),
    'l1_ratio':
    np.round(np.arange(0.01,1.0,0.05), 2),
    'tol': list(np.round(np.geomspace(1e-5,1e-3, 20),4))+list(np.round(np.linspace(1e-5,1e-3, 20),4)),
    'solver': ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
    'max_iter': np.arange(100, 300, 20)
}

for i in range(50):
  print(f'{i+1}th Model')
  params_trial = params.copy()
  for param in param_grid.keys():
    params_trial[param] = np.random.choice(param_grid[param])

  if params_trial['solver'] in ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag']:
    params_trial['l1_ratio'] = 0
  if params_trial['solver']=='liblinear':
    params_trial['penalty'] = np.random.choice(['l1', 'l2'])
  if params_trial['solver']=='saga':
    params_trial['penalty'] = 'elasticnet'
  print(params_trial)

  start_time = time.time()

  for label in target_cols:
    y_train_label = y_train[label]
    y_val_label = y_val[label]
    lg = LogisticRegression(
          C=params_trial['C'],
          l1_ratio=params_trial['l1_ratio'],
          tol=params_trial['tol'],
          class_weight=class_weight_dict[label],
          random_state=params_trial['random_state'],
          solver=params_trial['solver'],
          max_iter=params_trial['max_iter'],
          n_jobs=params_trial['n_jobs']
    )

    lg.fit(X_train, y_train_label)
    y_pred = lg.predict_proba(X_val)
    model_auc = average_precision_score(y_val_label, y_pred[:,1])

    if (model_auc > best_score[label][0][0]):
      print(f"Label {label} found new best params. New best score: {model_auc}!")
      best_score[label][0][0] = model_auc
      best_score[label][1] = params_trial
      best_score[label][2] = lg
      joblib.dump(best_score, '/content/drive/My Drive/bank_data/best_score.joblib')

    del lg
    gc.collect()

  print(f"Training Execution Time: {time.time() - start_time:.2f} seconds")











1th Model
{'penalty': 'elasticnet', 'C': np.float64(0.15874), 'l1_ratio': np.float64(0.76), 'tol': np.float64(0.0001), 'class_weight': None, 'random_state': 42, 'solver': np.str_('saga'), 'max_iter': np.int64(240), 'n_jobs': -1}
Label product_3_last_0 found new best params. New best score: 0.050205619555345284!
Label product_4_last_0 found new best params. New best score: 3.4005364113591936e-05!
Label product_5_last_0 found new best params. New best score: 0.0031707170887755!
Label product_6_last_0 found new best params. New best score: 0.0004467355676804385!
Label product_7_last_0 found new best params. New best score: 0.028478853404525484!
Label product_8_last_0 found new best params. New best score: 0.0009441048691756959!
Label product_9_last_0 found new best params. New best score: 0.0005111400071713426!
Label product_10_last_0 found new best params. New best score: 1.983995767475696e-05!
Label product_11_last_0 found new best params. New best score: 9.33256244850019e-05!
Label pro

1.6.1


In [ ]:
print(best_score)

NameError: name 'best_score' is not defined

In [ ]:
%whos


Variable                       Type         Data/Info
-----------------------------------------------------
col                            str          trend_pca_10
customer_info_cols             list         n=17
df_customer_info_target_cols   DataFrame             customer_code em<...>274038 rows x 53 columns]
drive                          module       <module 'google.colab.dri<...>s/google/colab/drive.py'>
filepath                       str          /content/drive/My Drive/b<...>n_dataset/chunk_9.parquet
gc                             module       <module 'gc' (built-in)>
i                              int          9
merged_train_dataset           str          /content/drive/My Drive/b<...>ata/merged_train_dataset/
new_buy_pca_parquet            str          /content/drive/My Drive/b<...>_data/new_buy_pca.parquet
np                             module       <module 'numpy' from '/us<...>kages/numpy/__init__.py'>
os                             module       <module 'os' (frozen)>
pa  